<div align="center">
<table><tr><td width="340" height="110" align="center">
<strong>Espacio para el logo de la universidad</strong><br>
Reemplaza este recuadro por la imagen cuando prepares la entrega.
</td></tr></table>

<h1>Proyecto de grado</h1>

<strong>Recuperación semántica y clasificación de la función de citas académicas en Computer Science</strong>

<p>Universidad de los Andes<br>Maestría en Inteligencia Artificial<br>Grupo 15</p>

<p>Camilo Bejarano · German Rodriguez · Jose Arteaga · Sebastian Toro</p>
</div>

## Introducción

En este notebook vamos a entrenar SciBERT para reconocer para qué se usa una cita
académica. Queremos comprobar si leer tres fragmentos del artículo citado ayuda más
que leer su título y resumen. Entrenaremos las dos entradas con los mismos ejemplos.

**Alcance de nuestros datos.** Usamos la entrega preparada para el Grupo 15,
con las exclusiones acordadas en la preparación de datos. Los insumos vienen de
A1–A3; en esta fase trabajamos con nuestra partición. Aprendemos y validamos con
las etiquetas automáticas de A2. Nuestro conjunto humano T_B tiene 2.968 citas
válidas y se utilizará para la prueba final, una vez fijados los modelos.
A partir de aquí hablaremos de entrenamiento, validación interna y prueba final.

Vamos a contar cómo llegamos a los datos de trabajo, mostrar sus clases, entrenar,
comprobar los modelos guardados y dejar evidencia en Drive y MLflow.

## 1. Preparamos la A100 y las bibliotecas

Primero comprobamos la GPU e instalamos las mismas versiones que venimos usando.
Ejecuta cada explicación y su celda de código en orden. Si Colab pide autorización
para Drive, será necesario aceptarla para leer los datos y guardar los resultados.


In [ ]:
import torch as _a4_gpu
if not _a4_gpu.cuda.is_available() or "A100" not in _a4_gpu.cuda.get_device_name(0):
    raise RuntimeError("Conecta una A100 de Colab antes de empezar esta corrida.")
print("GPU conectada:", _a4_gpu.cuda.get_device_name(0))
import subprocess
import sys

PINNED = [
    "transformers==4.57.1", "accelerate==1.11.0", "datasets==4.3.0",
    "scikit-learn==1.7.2", "matplotlib==3.10.7", "safetensors==0.6.2", "mlflow==3.17.2",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED])
print("Instalación terminada. Continúa si estas bibliotecas aún no estaban importadas.")


## 2. Conectamos Drive y damos nombre a nuestro experimento

Los textos siguen en `MAIA_PROYECTO`. La entrega training v2 está en
`MAIA_TESIS/TrainingData_Grupo15_v2`. También leeremos las listas de exclusión que
acompañan la preparación de nuestros datos. Todos los resultados de esta corrida
irán a una carpeta nueva.

Usaremos etiquetas automáticas A2 para aprender y validar. Si hay una desconexión, conservamos
el nombre de corrida y ponemos `RESUME=True` para recuperar lo ya guardado.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Fuera de Colab: configura rutas locales y REQUIRE_A100=False para pruebas.")

DATA_ROOT = Path("/content/drive/MyDrive/MAIA_PROYECTO")  # acceso directo: solo lectura
RESULTS_ROOT = Path("/content/drive/MyDrive/MAIA_TESIS")  # carpeta propia: resultados
assert DATA_ROOT.is_dir(), f"No se encuentra el acceso directo: {DATA_ROOT}"
assert RESULTS_ROOT.is_dir(), f"No se encuentra la carpeta de resultados: {RESULTS_ROOT}"
if RESULTS_ROOT.resolve().is_relative_to(DATA_ROOT.resolve()):
    raise ValueError("RESULTS_ROOT debe estar fuera del origen compartido DATA_ROOT.")
IDS_DIR = RESULTS_ROOT / "TrainingData_Grupo15_v2"
OLD_IDS_DIR = RESULTS_ROOT / "TrainingData_Grupo15"
A3_DIR = DATA_ROOT / "a3"
TRAINING_MANIFEST_SHA256 = "c7dd23f92b2272338a9d98dec7cf86476937658aa09a8cd3706c9402be34e231"
A3_MANIFEST_SHA256 = "30118fea362fc4c0389ab34e2460d566d886d5dd7f11e8a84483b62b98a7ce5b"
DATASET_RELEASE = "training_v2"
EXCLUSIONS_DIR = RESULTS_ROOT / "a4_inputs/g15_independent_v1"
TA_EXCLUSION_FILE = EXCLUSIONS_DIR / "excluded_ta_documents.json"
TA_EXCLUSION_SHA256 = "fd02b54442f8cf96381abed8e8ecbf6fa9830f4e34a05e85a75bbb861c4def9b"
FROZEN_MODEL_REVISION = "24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1"
BATCH_DIR = DATA_ROOT / "a2/batches_norw"
POOL_PATH = DATA_ROOT / "a2/pool_v11.1_ext/prelabels_pool.jsonl"  # opcional
GOLD_FILES = []  # no se incorporan etiquetas humanas al aprendizaje
GOLD_FINAL = True  # A3 cerró; esto NO habilita el test
RUN_TEST = False  # True únicamente para la evaluación final acordada; no para elegir modelos

VARIANT = "V2"  # V0, V1 o V2; conservar el mismo valor para ambas configuraciones
RUN_ID = "g15_a4_trainv2_prelabels_sinTA_seed42_01"
OUTPUT_ROOT = RESULTS_ROOT / "a4" / RUN_ID
CONFIGS = ["base", "fragmentos"]  # las dos entradas, desde el mismo SciBERT original
SEED = 42
SPLIT_SEED = "G15-A4-split-v1"
VAL_FRAC = 0.10
MAX_LENGTH = 512
CONTEXT_BUDGET = 192
EPOCHS = 4
LEARNING_RATE = 2e-5
TRAIN_BATCH = 16
GRAD_ACCUM = 2
EVAL_BATCH = 32
REQUIRE_A100 = True
RESUME = False
SNAPSHOT_INTERVAL_SECONDS = 300
LABELS = ["Background", "Gap", "Basis", "Comparison", "Application",
          "Improvement/Modification", "Evidence", "Identification of the Originator",
          "Further Reading"]
LABEL2ID = {v: k for k, v in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))
assert VARIANT == "V2" and GOLD_FINAL and not RUN_TEST, "Esta corrida entrena y valida; T_B queda cerrado."
assert CONFIGS == ["base", "fragmentos"]
assert not GOLD_FILES, "Esta corrida no admite etiquetas humanas para entrenar."
assert 0 < VAL_FRAC < 1 and 32 <= CONTEXT_BUDGET < MAX_LENGTH - 3
if RUN_TEST and (not GOLD_FILES or not GOLD_FINAL):
    raise ValueError("La prueba requiere gold final explícito. No usar pre-etiquetas como gold.")


## 3. Comprobamos la A100 y dejamos constancia del entorno

Primero revisamos la GPU y las versiones instaladas. Esto nos ayudará a repetir
la corrida y a entender posibles diferencias entre sesiones. Si falta la A100,
nos detenemos antes de gastar tiempo preparando el entrenamiento.

Guardamos también las decisiones y las huellas de los archivos. Si cambia la
entrega de datos, hará falta otra corrida; no la mezclaremos con una ya iniciada.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import collections
import gc
import hashlib
import importlib.metadata
import json
import random
import shutil
import time
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support, f1_score)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          EarlyStoppingCallback, set_seed)
from huggingface_hub import HfApi
from IPython.display import display

set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Activa GPU A100 en Colab antes de entrenar.")
GPU_NAME = torch.cuda.get_device_name(0)
if REQUIRE_A100 and "A100" not in GPU_NAME:
    raise RuntimeError(f"GPU asignada: {GPU_NAME}. Cambia a A100 o ajusta REQUIRE_A100.")
USE_BF16 = torch.cuda.is_bf16_supported()
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
# Cada modelo verificará sus propios checkpoints al llegar al entrenamiento.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def freeze_json(path, value):
    path = Path(path)
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != value:
        raise ValueError(f"RUN_ID ya tiene otra configuración o datos: {path.name}. Usa otro RUN_ID.")
    save_json(path, value)

def read_jsonl(path):
    with Path(path).open(encoding="utf-8-sig") as f:
        for n, line in enumerate(f, 1):
            if line.strip():
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(f"JSON inválido: {path}:{n}") from e
                if not isinstance(row, dict):
                    raise ValueError(f"Se esperaba un objeto: {path}:{n}")
                yield row

def unique_index(rows, name):
    index = {}
    for row in rows:
        key = row["inst_key"]
        if key in index:
            raise ValueError(f"inst_key repetida en {name}: {key}")
        index[key] = row
    return index

ENVIRONMENT = {"python": sys.version, "torch": torch.__version__,
               "cuda": torch.version.cuda, "gpu": GPU_NAME,
               "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
               "bf16": USE_BF16,
               "packages": {p.split("==")[0]: importlib.metadata.version(p.split("==")[0]) for p in PINNED}}
freeze_json(OUTPUT_ROOT / "request_manifest.json", {
    "dataset_release": DATASET_RELEASE, "a3_manifest_sha256": A3_MANIFEST_SHA256,
    "ta_exclusion_sha256": TA_EXCLUSION_SHA256, "label_policy": "A2_only",
    "variant": VARIANT, "seed": SEED, "split_seed": SPLIT_SEED, "val_frac": VAL_FRAC,
    "max_length": MAX_LENGTH, "context_budget": CONTEXT_BUDGET, "epochs": EPOCHS,
    "lr": LEARNING_RATE, "train_batch": TRAIN_BATCH, "grad_accum": GRAD_ACCUM,
    "eval_batch": EVAL_BATCH, "gold_final": GOLD_FINAL, "run_test": RUN_TEST,
    "ids_dir": str(IDS_DIR), "batch_dir": str(BATCH_DIR), "labels": LABELS,
    "gold_hashes": {str(p): sha256(p) for p in GOLD_FILES},
    "metadata_hashes": {p.name: sha256(p) for p in sorted(IDS_DIR.glob("*.json*"))},
    "environment": ENVIRONMENT,
})
save_json(OUTPUT_ROOT / "environment.json", ENVIRONMENT)
print(json.dumps(ENVIRONMENT, indent=2))


## 4. Comprobamos la entrega y reservamos la prueba humana

Verificamos que los archivos sean los de la entrega acordada y que cada cita tenga
un identificador único. Las huellas de los archivos nos permiten reconocer la
versión exacta que estamos usando.

También comprobamos el cierre de la revisión humana: hay 2.968 citas válidas y dos
ambiguas. En este paso solo verificamos la entrega y sus identificadores; todavía
no calculamos predicciones ni métricas sobre esas citas.


In [ ]:
"""Controles de A4 independiente: etiquetas A2 y exclusión de todo documento de T_A."""
import collections
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


def load_independent_release(ids_dir, a3_dir, old_ids_dir, exclusion_file, labels,
                             training_hash, metrics_hash, exclusion_hash):
    ids_dir, a3_dir, old_ids_dir = map(Path, [ids_dir, a3_dir, old_ids_dir])
    exclusion_file = Path(exclusion_file)
    hashes = {}
    manifest_path = checked_file(ids_dir, "manifest_training_v2_G15.json", training_hash, hashes)
    metrics_path = checked_file(a3_dir, "manifest_metrics.json", metrics_hash, hashes)
    checked_file(exclusion_file.parent, exclusion_file.name, exclusion_hash, hashes)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    exclusion = json.loads(exclusion_file.read_text(encoding="utf-8"))
    if (manifest["group"], manifest["test_set"], manifest["version"], manifest["variant"]) != ("G15", "B", "v2", "V0"):
        raise ValueError("La entrega no corresponde a training v2 de G15.")
    for filename, digest in manifest["files"].items():
        checked_file(ids_dir, filename, digest, hashes)
    rows = read_records(ids_dir / "training_v2_G15.jsonl")
    keyed_records(rows, "training v2 G15")
    if len(rows) != manifest["v2_rows"]:
        raise ValueError("El tamaño de la entrega no coincide con su manifiesto.")
    if not all(r["high_confidence"] and r["primary_pre_label"] in labels and r["label_source"] == "pre_label" for r in rows):
        raise ValueError("Esta corrida solo acepta las preetiquetas originales de A2.")
    if exclusion["purpose"] != "exclude_TA_documents_from_G15_training_and_validation" or exclusion["scope"] != "all_sampled_packages_including_unused_reserves_and_buffers":
        raise ValueError("La exclusión no cubre todos los paquetes de T_A, incluidas sus reservas.")
    ta_documents = set(exclusion["documents"])
    if len(ta_documents) != exclusion["document_count"] or len(ta_documents) != 5233 or exclusion["packages"] != 3510:
        raise ValueError("La lista de documentos de T_A está incompleta o tiene duplicados.")
    gold_path = checked_file(a3_dir, "test_set_gold.jsonl", metrics["files"]["test_set_gold.jsonl"], hashes)
    own = []
    counts = collections.Counter()
    unclear = 0
    # De las filas ajenas solo se lee el identificador de conjunto para ignorarlas.
    # Las etiquetas humanas propias se usan aquí exclusivamente para comprobar el cierre.
    with gold_path.open(encoding="utf-8-sig") as stream:
        for line in stream:
            if not line.strip():
                continue
            record = json.loads(line)
            if record["set"] != "B":
                continue
            label = record["gold_primary"]
            if type(record["in_eval"]) is not bool or record["in_eval"] != (label != "unclear"):
                raise ValueError("El cierre humano de T_B tiene una marca contradictoria.")
            if label not in labels + ["unclear"]:
                raise ValueError("Hay una categoría desconocida en T_B.")
            if record["in_eval"]:
                counts[label] += 1
            else:
                unclear += 1
            own.append({key: record[key] for key in ["inst_key", "group_key", "citing_paper_id", "cited_paper_id"]})
    keyed_records(own, "identificadores anotados de T_B")
    expected = {"Background": 330, "Gap": 310, "Basis": 322, "Comparison": 313,
                "Application": 400, "Improvement/Modification": 268, "Evidence": 386,
                "Identification of the Originator": 324, "Further Reading": 315}
    if dict(counts) != expected or sum(counts.values()) != metrics["in_eval"]["B"] or unclear != metrics["unclear"]["B"]:
        raise ValueError("T_B no coincide con el cierre documentado de A3.")
    old_manifest = json.loads((old_ids_dir / "manifest_training_v1_G15.json").read_text(encoding="utf-8"))
    checked_file(old_ids_dir, "test_set_ids_G15.jsonl", old_manifest["files"]["test_set_ids_G15.jsonl"], hashes)
    old_index = keyed_records(read_records(old_ids_dir / "test_set_ids_G15.jsonl"), "identificadores originales G15")
    for row in own:
        if row["inst_key"] not in old_index:
            raise ValueError("Una cita del cierre T_B no está en la muestra original.")
        row["source_batch"] = old_index[row["inst_key"]]["source_batch"]
    closure = {"date": "2026-11-09", "valid_test_rows": sum(counts.values()), "unclear_test_rows": unclear,
               "valid_test_counts": dict(counts), "test_scored": False, "training_uses_human_labels": False}
    return manifest, rows, own, ta_documents, hashes, closure, exclusion


def make_independent_labels(rows, labels):
    labeled = []
    for row in rows:
        if row.get("test_set") == "A" or row.get("label_source") != "pre_label" or row.get("gold_set") is not None:
            raise ValueError("Una fila de T_A o una etiqueta humana intentó entrar al entrenamiento.")
        label = row["primary_pre_label"]
        if label not in labels:
            raise ValueError("La preetiqueta no pertenece a las nueve clases.")
        labeled.append({**row, "label": label, "label_source": "pre_label"})
    return labeled


def audit_independent_splits(train, validation, test_ids, ta_documents):
    for name, rows in [("train", train), ("validation", validation)]:
        if document_set(rows) & ta_documents:
            raise ValueError(f"Un documento de T_A aparece en {name}.")
        if any(r.get("test_set") == "A" or r.get("label_source") != "pre_label" or r.get("gold_set") is not None for r in rows):
            raise ValueError(f"Hay una cita ajena o una etiqueta humana en {name}.")
    splits = [("train", train), ("validation", validation), ("test_reserved", test_ids)]
    for i, (name, rows) in enumerate(splits):
        for other_name, other in splits[i + 1:]:
            if document_set(rows) & document_set(other):
                raise ValueError(f"Hay documentos compartidos entre {name} y {other_name}.")


def plot_independent_eda(train, validation, labels, output_root):
    import matplotlib.pyplot as plt
    root = Path(output_root)
    frame = pd.DataFrame({"entrenamiento": collections.Counter(r["label"] for r in train),
                          "validacion": collections.Counter(r["label"] for r in validation)}).reindex(labels).fillna(0).astype(int)
    frame["total"] = frame.sum(axis=1)
    frame.index.name = "label"
    frame.to_csv(root / "eda_class_distribution.csv")
    totals = frame.total
    fig, (pie, bars) = plt.subplots(1, 2, figsize=(17, 6))
    wedges, _, _ = pie.pie(totals, startangle=90, colors=plt.get_cmap("tab10")(np.arange(len(labels))),
        autopct=lambda pct: f"{pct:.1f}%" if pct >= 3 else "", pctdistance=.78,
        wedgeprops={"width": .42, "edgecolor": "white"})
    pie.text(0, 0, f"{int(totals.sum()):,}\nejemplos", ha="center", va="center", fontsize=15)
    pie.set_title("Las clases de nuestros datos de trabajo")
    pie.legend(wedges, [f"{label}: {n:,} ({n / totals.sum():.1%})" for label, n in totals.items()],
        loc="upper center", bbox_to_anchor=(.5, -.02), fontsize=8, ncol=2)
    frame[["entrenamiento", "validacion"]].plot.barh(ax=bars, color=["#16877c", "#8fa6be"])
    bars.invert_yaxis()
    for container in bars.containers:
        bars.bar_label(container, fmt="%.0f", padding=3, fontsize=8)
    bars.set(xlabel="Número de ejemplos", ylabel="", title="Entrenamiento y validación por clase")
    bars.set_xlim(0, int(frame.entrenamiento.max()) * 1.22)
    fig.suptitle("A4 · Grupo 15 · etiquetas automáticas A2 · aislamiento por documento")
    fig.tight_layout()
    fig.savefig(root / "eda_class_distribution.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"1. Usaremos {len(train):,} ejemplos para entrenar y {len(validation):,} para validar; T_B queda reservado.")
    print(f"2. La clase más frecuente es {totals.idxmax()} ({totals.max()/totals.sum():.1%}); "
          f"la más escasa es {totals.idxmin()} ({int(totals.min()):,} en total, {int(frame.loc[totals.idxmin(),'validacion'])} en validación).")
    return frame


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_records(path):
    with Path(path).open(encoding="utf-8-sig") as stream:
        return [json.loads(line) for line in stream if line.strip()]

def keyed_records(rows, name):
    result = {}
    for row in rows:
        key = row["inst_key"]
        if key in result:
            raise ValueError(f"Hay una clave repetida en {name}: {key}")
        citing, para, span, cited = key.split("|")
        if citing != row["citing_paper_id"] or cited != row["cited_paper_id"] or citing == cited:
            raise ValueError(f"La clave no corresponde a sus artículos en {name}: {key}")
        if not row["group_key"].startswith(f"{citing}|{para}|"):
            raise ValueError(f"El grupo de citas no corresponde a la clave en {name}: {key}")
        result[key] = row
    return result

def checked_file(root, name, expected, hashes):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if path.parent != root:
        raise ValueError("El manifiesto contiene una ruta fuera de su carpeta.")
    actual = file_hash(path)
    if actual != expected:
        raise ValueError(f"El archivo cambió respecto de la entrega revisada: {name}")
    hashes[str(path)] = actual
    return path

def release_rules(rows):
    rules = {"citing": set(), "cited": set()}
    for row in rows:
        if row["group"] != "G15" or row["role"] not in rules:
            raise ValueError("La lista de exclusión no corresponde a G15 o tiene un rol desconocido.")
        rules[row["role"]].add(row["paper_id"])
    return rules

def release_excluded(row, rules):
    return row["citing_paper_id"] in rules["citing"] or row["cited_paper_id"] in rules["cited"]

def document_set(rows):
    return {r[key] for r in rows for key in ["citing_paper_id", "cited_paper_id"]}

manifest, train_v0, test_ids, ta_documents, input_hashes, a3_closure, ta_exclusion = load_independent_release(
    IDS_DIR, A3_DIR, OLD_IDS_DIR, TA_EXCLUSION_FILE, LABELS,
    TRAINING_MANIFEST_SHA256, A3_MANIFEST_SHA256, TA_EXCLUSION_SHA256)
save_json(OUTPUT_ROOT / "a3_closure.json", a3_closure)
save_json(OUTPUT_ROOT / "ta_exclusion_provenance.json", {key:value for key,value in ta_exclusion.items() if key != "documents"})
print("Entrega verificada:", len(train_v0), "filas; T_B reservado:", a3_closure["valid_test_rows"])
print("La lista para excluir T_A cubre", len(ta_documents), "documentos; no aporta etiquetas al modelo.")


## 5. Retiramos todo documento reservado

Aplicamos las listas de exclusión de la entrega a los dos artículos de cada cita:
el que cita y el citado. Así evitamos que un documento reservado termine entre
los ejemplos con los que aprenderemos o validaremos.

La tabla permite seguir cuántos ejemplos quedan por clase. Las reservas que A3
liberó y que nunca fueron anotadas pueden utilizarse si cumplen todos los
controles. Cada retiro queda contado y documentado.


In [ ]:

RULES = {v: release_rules(read_records(IDS_DIR / f"exclusion_list_G15_{v}_v2.jsonl"))
         for v in ["V0", "V1", "V2"]}
own_v2_rows = [r for r in train_v0 if not release_excluded(r,RULES["V2"])]
eligible = [dict(r) for r in own_v2_rows if r.get("test_set") != "A"
            and not {r["citing_paper_id"],r["cited_paper_id"]} & ta_documents]
if document_set(eligible) & document_set(test_ids):
    raise ValueError("Un documento de nuestro T_B sigue presente en los datos de trabajo.")
impact = pd.DataFrame({"V2 antes de excluir T_A":collections.Counter(r["primary_pre_label"] for r in own_v2_rows),
    "Datos independientes":collections.Counter(r["primary_pre_label"] for r in eligible)}).reindex(LABELS).fillna(0).astype(int)
impact.loc["TOTAL"] = impact.sum()
impact.to_csv(OUTPUT_ROOT / "isolation_impact.csv")
display(impact)
isolation_summary = {"dataset_release":DATASET_RELEASE,"variant":"V2","ta_scope":ta_exclusion["scope"],
    "v0_rows":len(train_v0),"after_own_V2":len(own_v2_rows),"removed_for_TA":len(own_v2_rows)-len(eligible),
    "eligible_rows":len(eligible),"ta_documents_in_eligible":len(document_set(eligible)&ta_documents),
    "tb_documents_in_eligible":len(document_set(eligible)&document_set(test_ids))}
save_json(OUTPUT_ROOT / "isolation_summary.json",isolation_summary)


## 6. Preparamos únicamente las etiquetas automáticas

Cada ejemplo conserva la preetiqueta de A2. Comprobamos su procedencia y que
pertenezca a una de nuestras nueve clases. Si aparece una etiqueta humana o una
cita fuera de la partición permitida, la ejecución se detiene para revisarla.

Este paso deja claro qué aprende el modelo y con qué referencias mediremos la
validación interna.


In [ ]:

labeled = make_independent_labels(eligible,LABELS)
test_rows = []
save_json(OUTPUT_ROOT / "label_provenance.json", {"dataset_release":DATASET_RELEASE,"label_policy":"A2_only",
    "pre_label_rows":len(labeled),"human_label_rows":0,"TA_rows":0,"reserved_test_set":"B","test_scored":False})
print("Todos los ejemplos de aprendizaje y validación conservan su etiqueta automática A2:",len(labeled))


## 7. Separamos los datos y miramos sus clases

Una regla fija por artículo citante reserva aproximadamente el 10% para validar.
Después quitamos los cruces de documentos entre aprendizaje y validación. También
comprobamos la separación con la prueba final, tanto para el citante como para el citado.

La torta muestra el total que realmente usaremos y las barras separan entrenamiento
y validación. Con los archivos actuales, Improvement/Modification queda con 46
ejemplos para aprender y solo 3 para validar. Su resultado de validación será muy
sensible a cada caso; no cambiaremos la partición buscando mejores puntajes.


In [ ]:
def validation_citing(citing_id):
    digest = hashlib.sha256(f"{SPLIT_SEED}|val|{citing_id}".encode()).hexdigest()
    return int(digest[:16], 16) / 2**64 < VAL_FRAC

val_rows = [r for r in labeled if validation_citing(r["citing_paper_id"])]
tr_before_purge = [r for r in labeled if not validation_citing(r["citing_paper_id"])]
val_citing = {r["citing_paper_id"] for r in val_rows}
val_cited = {r["cited_paper_id"] for r in val_rows}
if VARIANT == "V0":
    train_rows = tr_before_purge
elif VARIANT == "V1":
    train_rows = [r for r in tr_before_purge if r["cited_paper_id"] not in val_cited]
else:
    val_docs = val_citing | val_cited
    train_rows = [r for r in tr_before_purge if r["citing_paper_id"] not in val_docs
                  and r["cited_paper_id"] not in val_docs]
assert train_rows and val_rows
splits = {"train": train_rows, "validation": val_rows}
if RUN_TEST:
    splits["test"] = test_rows
for a, ar in splits.items():
    assert all(r["label"] in LABEL2ID for r in ar)
    if a != "test":
        assert not any(release_excluded(r, RULES[VARIANT]) for r in ar)
    for b, br in splits.items():
        if a >= b:
            continue
        for key in ("inst_key", "group_key", "citing_paper_id"):
            assert not ({r[key] for r in ar} & {r[key] for r in br}), (a, b, key)
        if VARIANT == "V1":
            assert not ({r["cited_paper_id"] for r in ar} & {r["cited_paper_id"] for r in br})
        if VARIANT == "V2":
            da = {r[k] for r in ar for k in ("citing_paper_id", "cited_paper_id")}
            db = {r[k] for r in br for k in ("citing_paper_id", "cited_paper_id")}
            assert not da & db
counts = pd.DataFrame({s: collections.Counter(r["label"] for r in rows)
                       for s, rows in splits.items()}).reindex(LABELS).fillna(0).astype(int)
if (counts[["train", "validation"]] == 0).any().any():
    raise ValueError("Alguna clase está ausente en train o val. Revisa el diseño del split.")
display(counts)
counts.to_csv(OUTPUT_ROOT / "split_class_counts.csv")
KEY_FIELDS = ["inst_key", "group_key", "citing_paper_id", "cited_paper_id", "label", "label_source", "source_batch"]
split_hashes = {}
for name, rows in splits.items():
    rows.sort(key=lambda r: r["inst_key"])
    path = OUTPUT_ROOT / f"split_{name}_ids.jsonl"
    content = "".join(json.dumps({k: r[k] for k in KEY_FIELDS}, ensure_ascii=False, sort_keys=True) + "\n" for r in rows)
    split_hashes[name] = hashlib.sha256(content.encode()).hexdigest()
    path.write_text(content, encoding="utf-8")
save_json(OUTPUT_ROOT / "split_summary.json", {
    "split_seed": SPLIT_SEED, "validation_fraction_target": VAL_FRAC,
    "removed_for_validation_isolation": len(tr_before_purge) - len(train_rows),
    "rows": {k: len(v) for k, v in splits.items()}, "hashes": split_hashes,
    "temporal_split": False, "section_stratification": False,
})

audit_independent_splits(train_rows, val_rows, test_ids, ta_documents)
eda_counts = plot_independent_eda(train_rows, val_rows, LABELS, OUTPUT_ROOT)
source_counts = pd.DataFrame({name: collections.Counter(row["label_source"] for row in rows)
                              for name, rows in [("train", train_rows), ("validation", val_rows)]}).fillna(0).astype(int)
source_counts.to_csv(OUTPUT_ROOT / "split_label_sources.csv")
display(source_counts)


## 8. Buscamos los textos que acompañan a las citas

Los archivos de entrenamiento contienen identificadores y etiquetas, no todos los
textos. Vamos a buscar sus contextos, títulos, resúmenes y fragmentos en los ocho
lotes de A2 que ya hemos usado. Mostramos los nombres de los campos para comprobar
que seguimos leyendo la estructura correcta. Si cambia, nos detenemos para revisarla.


In [ ]:
BATCH_FIELDS = {
    "inst_key": "inst_key", "context": "context",
    "title": "cited_title", "abstract": "cited_abstract", "chunks": "top3",
}
CHUNK_FIELDS = {"text": "text", "section": "section"}

def field_at(row, dotted):
    value = row
    for part in dotted.split("."):
        if not isinstance(value, dict) or part not in value:
            raise KeyError(f"Campo '{dotted}' ausente. Ajusta el adaptador con el esquema real.")
        value = value[part]
    return value

required_batches = sorted({r["source_batch"] for rows in splits.values() for r in rows})
batch_paths = {name: BATCH_DIR / f"{name}.jsonl" for name in required_batches}
missing = [str(p) for p in batch_paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Faltan lotes con texto; descárgalos de a2/batches_norw/:\n" + "\n".join(missing))
for name, path in batch_paths.items():
    first = next(read_jsonl(path), None)
    if first is None:
        raise ValueError(f"Lote vacío: {name}")
    print(name, {k: type(v).__name__ for k, v in first.items()})
    chunks = field_at(first, BATCH_FIELDS["chunks"])
    if isinstance(chunks, list) and chunks:
        print("  primer chunk:", {k: type(v).__name__ for k, v in chunks[0].items()})
freeze_json(OUTPUT_ROOT / "batch_manifest.json", {
    "hashes": {name: sha256(path) for name, path in batch_paths.items()},
    "batch_fields": BATCH_FIELDS, "chunk_fields": CHUNK_FIELDS,
})
print("Si el esquema coincide, continúa. Si difiere, corrige el adaptador y repite esta celda.")


## 9. Reunimos la información de cada ejemplo

Ahora unimos cada identificador con su texto exacto y comprobamos que tenga lo
necesario para las dos entradas. Cada contexto debe señalar una sola cita objetivo.
Si falta un texto o sus tres fragmentos, lo revisamos; no lo rellenamos ni buscamos
otro ejemplo parecido. Así la comparación usa las mismas citas en ambos modelos.


In [ ]:
def require_text(value, name):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"Texto vacío o tipo incorrecto: {name}")
    return value.strip()

def normalize_batch(row):
    out = {k: require_text(field_at(row, BATCH_FIELDS[k]), k)
           for k in ("inst_key", "context", "title", "abstract")}
    if out["context"].count("[[TARGET]]") != 1:
        raise ValueError(f"Se requiere un [[TARGET]] por contexto: {out['inst_key']}")
    if "{{cite:" in out["context"]:
        raise ValueError("Quedan marcadores crudos. Usa el renderizador congelado de A2.")
    raw_chunks = field_at(row, BATCH_FIELDS["chunks"])
    if not isinstance(raw_chunks, list) or len(raw_chunks) != 3:
        raise ValueError(f"Top-3 incompleto: {out['inst_key']}")
    out["chunks"] = []
    for ch in raw_chunks:
        text = require_text(field_at(ch, CHUNK_FIELDS["text"]), "chunk.text")
        section = field_at(ch, CHUNK_FIELDS["section"])
        if section is not None and not isinstance(section, str):
            raise ValueError(f"Tipo incorrecto de sección: {out['inst_key']}")
        missing_section = section is None or not section.strip()
        out["chunks"].append({"text": text, "section": "Unknown" if missing_section else section.strip(),
                              "section_missing": missing_section})
    return out

needed = {r["inst_key"]: r["source_batch"] for rows in splits.values() for r in rows}
texts = {}
for batch, path in batch_paths.items():
    input_hashes[str(path)] = sha256(path)
    seen_batch = set()
    for raw in read_jsonl(path):
        key = field_at(raw, BATCH_FIELDS["inst_key"])
        if key in seen_batch:
            raise ValueError(f"Clave duplicada en {batch}: {key}")
        seen_batch.add(key)
        if key in needed and needed[key] == batch:
            texts[key] = normalize_batch(raw)
missing_keys = sorted(needed.keys() - texts.keys())
if missing_keys:
    raise ValueError(f"Sin reconstruir: {len(missing_keys)}. Primeras claves: {missing_keys[:5]}")
assert len(texts) == len(needed)
save_json(OUTPUT_ROOT / "reconstruction_audit.json", {
    "required_rows": len(needed), "reconstructed_rows": len(texts),
    "batches": required_batches, "adapter": BATCH_FIELDS, "chunk_adapter": CHUNK_FIELDS,
    "rows_with_missing_section": sum(any(ch["section_missing"] for ch in ex["chunks"]) for ex in texts.values()),
    "chunks_with_missing_section": sum(ch["section_missing"] for ex in texts.values() for ch in ex["chunks"]),
})
section_rows = []
for split, rows in splits.items():
    cnt = collections.Counter(ch["section"] for r in rows for ch in texts[r["inst_key"]]["chunks"])
    section_rows.extend({"split": split, "section": sec, "chunks": n} for sec, n in cnt.items())
pd.DataFrame(section_rows).to_csv(OUTPUT_ROOT / "section_distribution.csv", index=False)
print(f"Reconstruidas {len(texts):,} instancias con ambas configuraciones completas.")


## 10. Preparamos los textos para SciBERT

SciBERT recibe piezas de texto llamadas tokens y tiene un espacio limitado por
ejemplo. Mantendremos el mismo reparto usado antes para comparar sin cambiar más
decisiones a la vez. La cita objetivo se conserva dentro del contexto.

Contaremos cuántos ejemplos necesitan recorte, tanto en el contexto como en la
información del citado. Ese recorte puede influir en el resultado, sobre todo al
repartir el espacio entre tres fragmentos. Quedará documentado junto al modelo.


In [ ]:
MODEL_ID = "allenai/scibert_scivocab_uncased"
MODEL_REVISION = FROZEN_MODEL_REVISION
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
MARKERS = ["[[TARGET]]", "[CITE]", "[FORMULA]", "[FIGURE]", "[TABLE]"]
tokenizer.add_special_tokens({"additional_special_tokens": MARKERS})
TARGET_ID = tokenizer.convert_tokens_to_ids("[[TARGET]]")

def fair_caps(lengths, budget):
    caps = [0] * len(lengths)
    remaining = budget
    while remaining and any(c < n for c, n in zip(caps, lengths)):
        for i, n in enumerate(lengths):
            if remaining and caps[i] < n:
                caps[i] += 1
                remaining -= 1
    return caps

def encode_example(example, config, tok, max_length=512, context_budget=192):
    ids = lambda s: tok.encode(s, add_special_tokens=False)
    context = ids(example["context"])
    target_id = tok.convert_tokens_to_ids("[[TARGET]]")
    if context.count(target_id) != 1:
        raise ValueError("El contexto debe contener exactamente un marcador objetivo.")
    target = context.index(target_id)
    start = max(0, min(target - context_budget // 2, len(context) - context_budget))
    context_cut = context[start:start + context_budget]
    assert tok.num_special_tokens_to_add(pair=True) == 3, "Se requiere tokenizador BERT."
    budget = max_length - 3 - len(context_cut)
    assert budget >= 32 and target_id in context_cut
    stats = {"context_original": len(context), "context_kept": len(context_cut)}
    if config == "base":
        title = ids("Title: " + example["title"])
        abstract = ids("Abstract: " + example["abstract"])
        title_kept = title[:min(64, budget // 3)]
        abstract_kept = abstract[:budget - len(title_kept)]
        evidence = title_kept + abstract_kept
        stats.update(evidence_original=len(title) + len(abstract), evidence_kept=len(evidence),
                     title_original=len(title), title_kept=len(title_kept),
                     abstract_original=len(abstract), abstract_kept=len(abstract_kept))
    elif config == "fragmentos":
        sections = [ids(f"Section: {ch['section']} Passage: ")[:24] for ch in example["chunks"]]
        bodies = [ids(ch["text"]) for ch in example["chunks"]]
        caps = fair_caps([len(h) + len(b) for h, b in zip(sections, bodies)], budget)
        evidence = []
        for i, (head, body, cap) in enumerate(zip(sections, bodies, caps), 1):
            kept = (head + body)[:cap]
            if len(kept) <= len(head):
                raise ValueError("Presupuesto insuficiente para conservar texto de cada chunk.")
            evidence.extend(kept)
            stats[f"chunk{i}_original"] = len(head) + len(body)
            stats[f"chunk{i}_kept"] = len(kept)
        stats.update(evidence_original=sum(len(h) + len(b) for h, b in zip(sections, bodies)),
                     evidence_kept=len(evidence))
    else:
        raise ValueError(config)
    input_ids = [tok.cls_token_id] + context_cut + [tok.sep_token_id] + evidence + [tok.sep_token_id]
    token_types = [0] * (len(context_cut) + 2) + [1] * (len(evidence) + 1)
    assert len(input_ids) <= max_length and len(token_types) == len(input_ids)
    stats["total_tokens"] = len(input_ids)
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
            "token_type_ids": token_types}, stats

encoded = {}
audit_rows = []
for config in CONFIGS:
    encoded[config] = {}
    for split, rows in splits.items():
        features = []
        for row in rows:
            features_row, stats = encode_example(texts[row["inst_key"]], config, tokenizer,
                                                  MAX_LENGTH, CONTEXT_BUDGET)
            features.append({**features_row, "labels": LABEL2ID[row["label"]]})
            audit_rows.append({"config": config, "split": split, "inst_key": row["inst_key"], **stats})
        encoded[config][split] = Dataset.from_list(features)
audit = pd.DataFrame(audit_rows)
audit["context_truncated"] = audit.context_kept < audit.context_original
audit["evidence_truncated"] = audit.evidence_kept < audit.evidence_original
audit.to_csv(OUTPUT_ROOT / "token_budget_audit.csv", index=False)
display(audit.groupby(["config", "split"])[["total_tokens", "context_truncated", "evidence_truncated"]].mean())


## 11. Fijamos nuestras decisiones y medimos una referencia sencilla

Guardamos la versión de datos, las exclusiones, las particiones y los ajustes de
SciBERT. El identificador nuevo evita mezclar esta corrida con las anteriores.
También medimos qué pasaría si siempre predijéramos la clase más frecuente.

Las dos entradas conservan semilla 42, hasta cuatro épocas y los mismos ajustes.
Las decisiones se eligen con la validación interna. Las pocas muestras de algunas clases limitan lo
que podemos concluir de pequeñas diferencias en validación.


In [ ]:
run_spec = {
    "group": "G15", "test_set": "B", "isolation": VARIANT,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "seed": SEED,
    "split_seed": SPLIT_SEED, "split_hashes": split_hashes, "input_hashes": input_hashes,
    "labels": LABELS, "markers": MARKERS, "max_length": MAX_LENGTH,
    "context_budget": CONTEXT_BUDGET, "title_budget": 64, "section_budget": 24,
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "train_batch": TRAIN_BATCH,
    "gradient_accumulation": GRAD_ACCUM, "eval_batch": EVAL_BATCH,
    "weight_decay": 0.01, "warmup_ratio": 0.1, "early_stopping_patience": 2,
    "loss": "unweighted_cross_entropy", "gold_final": GOLD_FINAL, "run_test": RUN_TEST,
    "batch_fields": BATCH_FIELDS, "chunk_fields": CHUNK_FIELDS, "environment": ENVIRONMENT,
}
run_spec.update(dataset_release=DATASET_RELEASE,label_policy="A2_only",training_uses_human_labels=False,
    ta_exclusion_sha256=TA_EXCLUSION_SHA256,ta_exclusion_scope=ta_exclusion["scope"],
    ta_document_count=len(ta_documents),a3_manifest_sha256=A3_MANIFEST_SHA256,
    purpose="G15_independent_training_validation",reserved_test_set="B")
spec_path = OUTPUT_ROOT / "run_manifest.json"
if spec_path.exists():
    old_spec = json.loads(spec_path.read_text(encoding="utf-8"))
    if old_spec != run_spec:
        raise ValueError("RUN_ID ya existe con otro manifiesto. Usa otro RUN_ID.")
save_json(spec_path, run_spec)
majority = collections.Counter(r["label"] for r in train_rows).most_common(1)[0][0]
y_val = [LABEL2ID[r["label"]] for r in val_rows]
baseline_metrics = {"class": majority,
    "validation_macro_f1": f1_score(y_val, [LABEL2ID[majority]] * len(y_val),
                                     labels=list(ID2LABEL), average="macro", zero_division=0)}
save_json(OUTPUT_ROOT / "majority_baseline.json", baseline_metrics)
print(baseline_metrics)


## 12. Registramos esta fase independiente en MLflow

Esta corrida tendrá su propio nombre y estará en el experimento
`G15_A4_SciBERT_Independiente`. El historial anterior se conserva como registro.
No se reutilizan modelos ni resultados anteriores para entrenar o seleccionar aquí.

MLflow guarda las decisiones, curvas y reportes. Los modelos y puntos de recuperación
quedan en Drive. Se respaldan al empezar, al guardar épocas y al cerrar; durante el
entrenamiento se comprueba además cada cinco minutos si corresponde un respaldo.
Solo una sesión debe escribir el historial a la vez.


In [ ]:
"""Se incrusta en el notebook: no requiere subir este archivo a Colab."""
import contextlib
import datetime as dt
import math
import numbers
import sqlite3
import tempfile
import uuid
import zipfile
from pathlib import PurePosixPath
import mlflow
from mlflow import MlflowClient
from transformers import TrainerCallback


def tracking_restore(root, snapshots):
    """Solo restaura antes de abrir MLflow; nunca sobrescribe una DB local."""
    root.mkdir(parents=True, exist_ok=True)
    if (root / "mlflow.db").exists():
        return
    candidates = sorted((p.parent for p in snapshots.glob("*/COMPLETE.json")),
        key=lambda p: dt.datetime.fromisoformat(json.loads((p / "COMPLETE.json").read_text())["created_at_utc"]))
    if not candidates:
        print("Sin historial previo: se creará una base MLflow nueva.")
        return
    source = candidates[-1]
    receipt = json.loads((source / "COMPLETE.json").read_text())
    archive_path = source / "snapshot.zip"
    assert receipt["schema_version"] == 1
    assert receipt["snapshot_id"] == source.name
    assert receipt["mlflow_version"] == mlflow.__version__
    assert archive_path.stat().st_size == receipt["zip_size"]
    assert sha256(archive_path) == receipt["zip_sha256"]
    assert (source / "snapshot.sha256").read_text().split()[0] == receipt["zip_sha256"]
    # Validar todo antes de copiar a las rutas canónicas.
    with tempfile.TemporaryDirectory() as tmp:
        target = Path(tmp)
        with zipfile.ZipFile(archive_path) as archive:
            names = archive.namelist()
            assert len(names) == len(set(names))
            manifest = json.loads(archive.read("snapshot_manifest.json"))
            assert manifest["schema_version"] == 1
            assert manifest["snapshot_id"] == receipt["snapshot_id"]
            assert manifest["canonical_root"] == str(root)
            assert manifest["mlflow_version"] == mlflow.__version__
            assert set(names) == set(manifest["files"]) | {"snapshot_manifest.json"}
            for entry in archive.infolist():
                path = PurePosixPath(entry.filename)
                assert not path.is_absolute() and ".." not in path.parts
                assert "\\" not in entry.filename and ":" not in entry.filename
                assert (entry.external_attr >> 16) & 0o170000 != 0o120000
                assert entry.filename in {"mlflow.db", "snapshot_manifest.json"} or entry.filename.startswith("artifacts/")
            archive.extractall(target)
        for name, metadata in manifest["files"].items():
            path = target / name
            assert path.stat().st_size == metadata["size"] and sha256(path) == metadata["sha256"]
        with contextlib.closing(sqlite3.connect(target / "mlflow.db")) as db:
            assert db.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
        assert not (root / "artifacts").exists(), "Hay artefactos locales sin DB; usa una sesión Colab limpia."
        if (target / "artifacts").exists():
            shutil.copytree(target / "artifacts", root / "artifacts")
        shutil.copyfile(target / "mlflow.db", root / "mlflow.db")
    print("Historial restaurado:", source.name)


class A4Tracking:
    def __init__(self, root, snapshots, output, run_id, interval=300):
        self.root, self.snapshots, self.output = Path(root), Path(snapshots), Path(output)
        self.run_id, self.interval = run_id, interval
        self.last_backup = 0
        self.parents, self.attempts = {}, {}
        self.snapshots.mkdir(parents=True, exist_ok=True)
        tracking_restore(self.root, self.snapshots)
        mlflow.set_tracking_uri("sqlite:///" + (self.root / "mlflow.db").as_posix())
        self.client = MlflowClient()
        name = "G15_A4_SciBERT_Independiente"
        experiment = self.client.get_experiment_by_name(name)
        self.experiment = (experiment.experiment_id if experiment else
            self.client.create_experiment(name, artifact_location=(self.root / "artifacts/scibert").as_uri()))
        # Solo el mismo RUN_ID: las sesiones anteriores terminaron sin cierre.
        previous = self.client.search_runs([self.experiment], filter_string=f"tags.a4_run_id = '{run_id}'")
        for run in previous:
            if run.info.status == "RUNNING":
                self.client.set_terminated(run.info.run_id, status="KILLED")
        self.backup(force=True)

    def backup(self, force=False):
        if not force and time.monotonic() - self.last_backup < self.interval:
            return
        created = dt.datetime.now(dt.timezone.utc)
        sid = created.strftime("%Y%m%dT%H%M%S%fZ_") + uuid.uuid4().hex[:8]
        with tempfile.TemporaryDirectory() as tmp:
            stage = Path(tmp) / "stage"
            stage.mkdir()
            with contextlib.closing(sqlite3.connect(self.root / "mlflow.db")) as source:
                with contextlib.closing(sqlite3.connect(stage / "mlflow.db")) as dest:
                    source.backup(dest)
                    assert dest.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
            if (self.root / "artifacts").exists():
                shutil.copytree(self.root / "artifacts", stage / "artifacts")
            paths = sorted(p for p in stage.rglob("*") if p.is_file())
            assert not any(p.is_symlink() for p in stage.rglob("*"))
            manifest = dict(schema_version=1, snapshot_id=sid, created_at_utc=created.isoformat(),
                mlflow_version=mlflow.__version__, canonical_root=str(self.root), purpose="scibert_training",
                files={p.relative_to(stage).as_posix(): {"sha256": sha256(p), "size": p.stat().st_size} for p in paths})
            save_json(stage / "snapshot_manifest.json", manifest)
            packed = Path(tmp) / "snapshot.zip"
            with zipfile.ZipFile(packed, "w", zipfile.ZIP_DEFLATED) as archive:
                for path in sorted(p for p in stage.rglob("*") if p.is_file()):
                    archive.write(path, path.relative_to(stage).as_posix())
            digest = sha256(packed)
            destination = self.snapshots / sid
            destination.mkdir(exist_ok=False)
            shutil.copyfile(packed, destination / "snapshot.zip")
            assert sha256(destination / "snapshot.zip") == digest
            (destination / "snapshot.sha256").write_text(f"{digest}  snapshot.zip\n")
            save_json(destination / "COMPLETE.json", dict(schema_version=1, snapshot_id=sid,
                created_at_utc=created.isoformat(), mlflow_version=mlflow.__version__,
                zip_sha256=digest, zip_size=packed.stat().st_size))
        self.last_backup = time.monotonic()
        print("MLflow respaldado:", destination)

    def parent(self, config):
        if config not in self.parents:
            runs = self.client.search_runs([self.experiment],
                filter_string=f"tags.a4_run_id = '{self.run_id}' and tags.config = '{config}' and tags.kind = 'configuration'")
            if len(runs) > 1:
                raise ValueError("Más de un padre para la misma configuración")
            self.parents[config] = (runs[0].info.run_id if runs else self.client.create_run(self.experiment,
                tags={"mlflow.runName": f"{self.run_id}/{config}", "a4_run_id": self.run_id,
                      "config": config, "kind": "configuration"}).info.run_id)
        return self.parents[config]

    def metrics(self, run, values, step=0):
        for key, value in values.items():
            if isinstance(value, numbers.Real) and math.isfinite(float(value)):
                self.client.log_metric(run, key, float(value), step=int(step))

    def reports(self, config, run=None):
        run = run or self.parent(config)
        for directory, artifact_path in [(self.output, "provenance"), (self.output / config, "reports")]:
            for path in sorted(directory.iterdir()):
                if path.is_file() and path.suffix in {".json", ".csv", ".png", ".txt"}:
                    self.client.log_artifact(run, str(path), artifact_path=artifact_path)

    @contextlib.contextmanager
    def evaluation(self, config, directory, policy="v2_fp32"):
        parent = self.parent(config)
        rid = self.client.create_run(self.experiment, tags={
            "mlflow.runName": f"{config}/evaluation_{policy}_{uuid.uuid4().hex[:8]}",
            "mlflow.parentRunId": parent, "a4_run_id": self.run_id, "config": config,
            "kind": "evaluation", "evaluation_policy": policy,
            "canonical_precision": "fp32", "drive_output": str(directory)}).info.run_id
        try:
            yield rid
            self.client.log_artifacts(rid, str(directory), artifact_path="evaluation")
            self.client.set_terminated(rid, "FINISHED")
            self.client.set_tag(parent, "latest_evaluation_run", rid)
            self.backup(force=True)
        except BaseException as error:
            self.client.set_tag(rid, "failure", f"{type(error).__name__}: {error}"[:2000])
            self.client.set_terminated(rid, "KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            try:
                self.backup(force=True)
            except Exception as backup_error:
                print("Falló el respaldo de evaluación:", backup_error)
            raise

    @contextlib.contextmanager
    def attempt(self, config, spec, checkpoint):
        parent = self.parent(config)
        self.client.update_run(parent, status="RUNNING")
        rid = self.client.create_run(self.experiment, tags={"mlflow.runName": f"{config}/attempt_{uuid.uuid4().hex[:8]}",
            "mlflow.parentRunId": parent, "a4_run_id": self.run_id, "config": config,
            "kind": "attempt", "validation_reference": "prelabels_and_available_gold"}).info.run_id
        self.attempts[config] = rid
        try:
            (self.output / config).mkdir(parents=True, exist_ok=True)
            for key, value in spec.items():
                if not isinstance(value, (dict, list)):
                    self.client.log_param(rid, key, str(value)[:5000])
            self.client.log_param(rid, "config", config)
            self.client.log_param(rid, "resume_checkpoint", str(checkpoint))
            self.client.set_tag(rid, "drive_output", str(self.output / config))
            self.reports(config, rid)
            self.backup(force=True)
            yield rid
        except BaseException as error:
            self.client.set_tag(rid, "failure", f"{type(error).__name__}: {error}"[:2000])
            self.client.set_terminated(rid, "KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            self.client.set_terminated(parent, "FAILED")
            try:
                self.backup(force=True)
            except Exception as backup_error:
                print("Falló el respaldo de cierre:", backup_error)
            raise
        else:
            self.reports(config, rid)
            self.client.set_terminated(rid, "FINISHED")
            self.client.set_terminated(parent, "FINISHED")
            self.backup(force=True)


def commit_checkpoint(path):
    path = Path(path)
    required = ["trainer_state.json", "optimizer.pt", "scheduler.pt", "training_args.bin"]
    assert all((path / name).is_file() for name in required)
    assert list(path.glob("*safetensors")) or list(path.glob("pytorch_model*.bin"))
    assert list(path.glob("rng_state*.pth")), "Falta el estado aleatorio para reanudar"
    files = {p.name: {"sha256": sha256(p), "size": p.stat().st_size} for p in path.iterdir() if p.is_file() and p.name != "A4_COMPLETE.json"}
    save_json(path / "A4_COMPLETE.json", {"files": files})


def verified_checkpoint(directory):
    directory = Path(directory)
    candidates = sorted(directory.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]), reverse=True)
    for path in candidates:
        if not (path / "A4_COMPLETE.json").exists():
            print("Se omite checkpoint sin cierre:", path)
            continue
        files = json.loads((path / "A4_COMPLETE.json").read_text())["files"]
        if all((path / name).is_file() and (path / name).stat().st_size == meta["size"] and sha256(path / name) == meta["sha256"] for name, meta in files.items()):
            # Trainer vuelve a cargar también el mejor checkpoint previo.
            state = json.loads((path / "trainer_state.json").read_text())
            best = state.get("best_model_checkpoint")
            if best and not Path(best).is_dir():
                raise ValueError(f"Falta el mejor checkpoint requerido: {best}")
            if best and Path(best) != path:
                marker = Path(best) / "A4_COMPLETE.json"
                assert marker.is_file(), "El mejor checkpoint previo no tiene cierre"
                best_files = json.loads(marker.read_text())["files"]
                assert all((Path(best) / name).is_file() and sha256(Path(best) / name) == meta["sha256"]
                           for name, meta in best_files.items()), "Mejor checkpoint previo corrupto"
            return str(path)
        print("Se omite checkpoint con hashes inválidos:", path)
    if candidates:
        raise ValueError("Hay checkpoints pero ninguno completo. No se reinicia silenciosamente.")
    return None


class A4MLflowCallback(TrainerCallback):
    def __init__(self, tracking, run):
        self.tracking, self.run = tracking, run

    def on_log(self, args, state, control, logs=None, **kwargs):
        self.tracking.metrics(self.run, logs or {}, state.global_step)

    def on_step_end(self, args, state, control, **kwargs):
        self.tracking.backup()

    def on_save(self, args, state, control, **kwargs):
        path = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        commit_checkpoint(path)
        self.tracking.client.set_tag(self.run, "latest_complete_checkpoint", str(path))
        self.tracking.client.log_artifact(self.run, str(path / "A4_COMPLETE.json"),
            artifact_path=f"checkpoints/{path.name}")
        self.tracking.backup(force=True)


import re
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID), "RUN_ID: usa letras, números, _ y -."
tracking = A4Tracking(Path("/opt/maia/a4_tracking"), RESULTS_ROOT / "a4/_tracking/snapshots",
                      OUTPUT_ROOT, RUN_ID, SNAPSHOT_INTERVAL_SECONDS)
print("MLflow listo:", tracking.experiment, "en", mlflow.get_tracking_uri())

for config in CONFIGS:
    parent_id = tracking.parent(config)
    for key,value in {"dataset_release":DATASET_RELEASE,"label_policy":"A2_only","isolation":"V2_plus_all_TA_documents",
        "training_uses_human_labels":"false","test_scored":"false","scope":"G15_A4_independent"}.items():
        tracking.client.set_tag(parent_id,key,value)


## 13. Entrenamos primero base y después Top-3

Primero entrenamos con título y resumen; después empezamos de nuevo desde SciBERT
original para entrenar con los tres fragmentos. El segundo modelo no continúa lo
que aprendió el primero. Ambos usan los mismos ejemplos y ajustes.

Al cerrar cada época medimos la validación. Conservamos el punto con mejor F1 macro,
una medida que da el mismo peso a las nueve clases, y detenemos antes si deja de
mejorar. Cada modelo puede completar hasta cuatro épocas. Guardamos lo necesario
para retomarlo si Colab se desconecta y para usarlo después desde una aplicación.


In [ ]:
import inspect
from transformers.trainer_utils import get_last_checkpoint

def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    pred = np.asarray(logits).argmax(axis=-1)
    y = prediction.label_ids
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=list(ID2LABEL),
                                               average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, pred), "precision_micro": accuracy_score(y, pred),
            "recall_micro": accuracy_score(y, pred), "precision_macro": p,
            "recall_macro": r, "f1_macro": f,
            "f1_micro": f1_score(y, pred, labels=list(ID2LABEL), average="micro", zero_division=0)}

INFERENCE_TEMPLATE = """import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

__ENCODER__

class CitationClassifier:
    def __init__(self, model_dir, device=None):
        self.root = Path(model_dir)
        self.policy = json.loads((self.root / "input_policy.json").read_text(encoding="utf-8"))
        self.tokenizer = AutoTokenizer.from_pretrained(self.root, local_files_only=True)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = AutoModelForSequenceClassification.from_pretrained(self.root, local_files_only=True, attn_implementation="eager").to(self.device).eval()

    def predict(self, example):
        features, stats = encode_example(example, self.policy["config"], self.tokenizer,
                                         self.policy["max_length"], self.policy["context_budget"])
        batch = {k: torch.tensor([v], device=self.device) for k, v in features.items()}
        with torch.inference_mode():
            probs = self.model(**batch).logits.float().softmax(-1)[0].cpu().tolist()
        best = max(range(len(probs)), key=probs.__getitem__)
        return {"label": self.model.config.id2label[best], "score": probs[best],
                "scores": {self.model.config.id2label[i]: p for i, p in enumerate(probs)},
                "score_calibrated": False, "token_audit": stats}
"""

# IPython conserva el código fuente de funciones definidas en sus celdas.
ENCODER_SOURCE = inspect.getsource(fair_caps) + "\n" + inspect.getsource(encode_example)
validation_results = []
for config in CONFIGS:
    workdir = OUTPUT_ROOT / config
    checkpoint_dir = workdir / "checkpoints"
    if RESUME and (workdir / "TRAINING_COMPLETE.json").exists():
        manifest = json.loads((workdir / "TRAINING_COMPLETE.json").read_text())
        for name, digest in manifest["model_hashes"].items():
            assert sha256(workdir / "model" / name) == digest
        validation_results.append(json.loads((workdir / "validation_metrics.json").read_text()))
        parent_id = tracking.parent(config)
        tracking.client.set_tag(parent_id, "drive_output", str(workdir))
        tracking.reports(config)
        tracking.client.set_terminated(parent_id, "FINISHED")
        tracking.backup(force=True)
        print(config, "ya entrenado y verificado; se omite entrenamiento.")
        continue
    last = verified_checkpoint(checkpoint_dir) if checkpoint_dir.exists() else None
    if (last or (workdir / "model/config.json").exists()) and not RESUME:
        raise ValueError(f"{config} ya tiene entrenamiento. Usa otro RUN_ID o RESUME=True.")
    with tracking.attempt(config, run_spec, last) as attempt_id:
        set_seed(SEED)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, num_labels=len(LABELS),
            id2label=ID2LABEL, label2id=LABEL2ID, problem_type="single_label_classification",
            attn_implementation="eager",
        )
        model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
        args = TrainingArguments(
            output_dir=str(checkpoint_dir), num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE, per_device_train_batch_size=TRAIN_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM, per_device_eval_batch_size=EVAL_BATCH,
            weight_decay=0.01, warmup_ratio=0.1, lr_scheduler_type="linear",
            eval_strategy="epoch", save_strategy="epoch", logging_strategy="steps", logging_steps=50,
            load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
            save_total_limit=2, save_safetensors=True, seed=SEED, data_seed=SEED,
            restore_callback_states_from_checkpoint=True,
            bf16=USE_BF16, fp16=not USE_BF16, tf32=False, full_determinism=True,
            dataloader_num_workers=2, dataloader_pin_memory=True,
            optim="adamw_torch", report_to="none", push_to_hub=False,
        )
        trainer = Trainer(model=model, args=args, train_dataset=encoded[config]["train"],
            eval_dataset=encoded[config]["validation"], processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2), A4MLflowCallback(tracking, attempt_id)])
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        train_result = trainer.train(resume_from_checkpoint=last if RESUME and last else None)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        val_metrics = trainer.evaluate()
        export_dir = workdir / "model"
        trainer.save_model(str(export_dir))
        tokenizer.save_pretrained(export_dir)
        policy = {"config": config, "max_length": MAX_LENGTH, "context_budget": CONTEXT_BUDGET,
                  "markers": MARKERS, "model_revision": MODEL_REVISION, "score_calibrated": False}
        save_json(export_dir / "input_policy.json", policy)
        save_json(export_dir / "label_mapping.json", {"label2id": LABEL2ID, "id2label": ID2LABEL})
        save_json(export_dir / "run_manifest.json", run_spec)
        (export_dir / "inference.py").write_text(INFERENCE_TEMPLATE.replace("__ENCODER__", ENCODER_SOURCE), encoding="utf-8")
        contract = {"config": config, "required_input": ["context", "title", "abstract"] if config == "base"
                    else ["context", "chunks: lista de 3 objetos {text, section}"],
                    "context_target": "Exactamente un [[TARGET]]; otros marcadores según guía rev3.1",
                    "output": {"label": "una de 9 clases", "score": "softmax sin calibrar", "scores": "9 probabilidades"}}
        save_json(export_dir / "api_contract.json", contract)
        result = {"config": config, **val_metrics, "elapsed_session_seconds": elapsed,
                  "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
                  "best_checkpoint": trainer.state.best_model_checkpoint,
                  "best_validation_macro_f1": trainer.state.best_metric,
                  "train_metrics": train_result.metrics}
        save_json(workdir / "validation_metrics.json", result)
        save_json(workdir / "training_log.json", trainer.state.log_history)
        model_hashes = {p.relative_to(export_dir).as_posix(): sha256(p)
                        for p in export_dir.rglob("*") if p.is_file()}
        save_json(workdir / "TRAINING_COMPLETE.json", {"model_hashes": model_hashes,
                  "model_dir": str(export_dir), "attempt_id": attempt_id})
        tracking.metrics(attempt_id, val_metrics, trainer.state.global_step)
        tracking.metrics(attempt_id, {"elapsed_session_seconds": elapsed,
            "peak_gpu_allocated_gib": result["peak_gpu_allocated_gib"]}, trainer.state.global_step)
        validation_results.append(result)
        print(config, "exportado en", export_dir, "F1 macro val:", val_metrics["eval_f1_macro"])
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
display(pd.DataFrame(validation_results).drop(columns=["train_metrics"]))


## 14. Comprobamos los modelos guardados y medimos la validación

Queremos comprobar que el modelo exportado responda igual que el punto de
entrenamiento seleccionado. Después calculamos las métricas de las dos entradas
en la misma validación y miramos sus errores por categoría.

La comparación principal usa FP32 para mantener la misma forma de medir que en las
corridas anteriores. Puede haber pequeñas diferencias frente a los números que
aparecieron durante el entrenamiento; esta comprobación las deja registradas.


In [ ]:
"""Evaluación del modelo exportado; se incrusta en la sección 12 del notebook."""
from pathlib import Path
from types import SimpleNamespace
import gc
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments


def probability_metrics(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=-1)
    classes = list(range(probabilities.shape[1]))
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=classes, average="macro", zero_division=0)
    return {"accuracy": float(accuracy_score(labels, predictions)),
            "precision_macro": float(precision), "recall_macro": float(recall),
            "f1_macro": float(f1), "f1_micro": float(f1_score(
                labels, predictions, labels=classes, average="micro", zero_division=0))}


def compare_probabilities(reference, actual, keys, atol=2e-5, rtol=1e-4):
    reference, actual = np.asarray(reference), np.asarray(actual)
    if reference.shape != actual.shape or reference.ndim != 2 or len(keys) != len(actual) or not len(actual):
        raise ValueError("Las predicciones no tienen las mismas filas y clases")
    for values in [reference, actual]:
        if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
            raise ValueError("Probabilidades inválidas")
        if not np.allclose(values.sum(axis=1), 1, atol=1e-5):
            raise ValueError("Las probabilidades no suman uno")
    changed = reference.argmax(-1) != actual.argmax(-1)
    return {"n": len(keys), "atol": atol, "rtol": rtol,
            "probabilities_close": bool(np.allclose(reference, actual, atol=atol, rtol=rtol)),
            "max_absolute_difference": float(np.abs(reference - actual).max()),
            "label_disagreements": int(changed.sum()),
            "changed_inst_keys": [key for key, differs in zip(keys, changed) if differs]}


def predict_export_fp32(classifier, data, batch_size):
    classifier.model.float().eval()
    collator = DataCollatorWithPadding(classifier.tokenizer, pad_to_multiple_of=8)
    probabilities = []
    for start in range(0, len(data), batch_size):
        items = [{k: v for k, v in data[i].items() if k != "labels"}
                 for i in range(start, min(start + batch_size, len(data)))]
        batch = {k: v.to(classifier.device) for k, v in collator(items).items()}
        # Explícito incluso si el llamador usa un contexto autocast externo.
        with torch.inference_mode(), torch.autocast(device_type=torch.device(classifier.device).type, enabled=False):
            probabilities.extend(classifier.model(**batch).logits.float().softmax(-1).cpu().tolist())
    return np.asarray(probabilities)


def audit_export(classifier, data, keys, checkpoint, output_dir, batch_size, seed, mixed_precision):
    """Compara checkpoint/exportación en FP32 y reproduce la ruta de Trainer.

    Nunca entrena ni modifica el checkpoint o la exportación existente.
    Si falta el checkpoint, la auditoría de exportación queda explícitamente pendiente.
    """
    if mixed_precision not in {"bf16", "fp16", "fp32"}:
        raise ValueError("Precisión de Trainer no reconocida")
    if classifier.model.config._attn_implementation != "eager":
        raise ValueError("La comparación requiere atención eager en ambas rutas")
    probabilities = predict_export_fp32(classifier, data, batch_size)
    has_checkpoint = bool(checkpoint and Path(checkpoint).is_dir())
    source = Path(checkpoint) if has_checkpoint else classifier.root
    reference_model = AutoModelForSequenceClassification.from_pretrained(
        source, local_files_only=True, attn_implementation="eager").float().to(classifier.device).eval()
    try:
        audit = {"evaluation_policy": "v2_fp32", "canonical_precision": "fp32", "attention_implementation": "eager",
                 "trainer_precision": mixed_precision, "reference_source": str(source),
                 "export_check": "pending_checkpoint_missing"}
        if has_checkpoint:
            reference = SimpleNamespace(model=reference_model, tokenizer=classifier.tokenizer, device=classifier.device)
            expected = predict_export_fp32(reference, data, batch_size)
            comparison = compare_probabilities(expected, probabilities, keys)
            audit["checkpoint_vs_export_fp32"] = comparison
            if not comparison["probabilities_close"] or comparison["label_disagreements"]:
                raise ValueError(f"La exportación no reproduce el checkpoint: {comparison}")
            audit["export_check"] = "passed_full_validation"
        args = TrainingArguments(output_dir=str(Path(output_dir) / "trainer_prediction"),
            per_device_eval_batch_size=batch_size, report_to="none", seed=seed,
            use_cpu=torch.device(classifier.device).type == "cpu",
            bf16=mixed_precision == "bf16", fp16=mixed_precision == "fp16", tf32=False,
            dataloader_num_workers=0, disable_tqdm=False)
        trainer = Trainer(model=reference_model, args=args, processing_class=classifier.tokenizer,
            data_collator=DataCollatorWithPadding(classifier.tokenizer, pad_to_multiple_of=8))
        prediction = trainer.predict(data)
        logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
        trainer_probabilities = torch.as_tensor(logits).float().softmax(-1).numpy()
        audit["trainer_vs_export_fp32"] = compare_probabilities(trainer_probabilities, probabilities, keys)
        return probabilities, trainer_probabilities, audit
    finally:
        del reference_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

import importlib.util
import matplotlib.pyplot as plt

EVALUATION_ROOT = OUTPUT_ROOT / "evaluations" / ("v2_fp32_" + uuid.uuid4().hex[:12])
EVALUATION_ROOT.mkdir(parents=True, exist_ok=False)

def load_export(config):
    root = OUTPUT_ROOT / config / "model"
    spec = importlib.util.spec_from_file_location(f"a4_inference_{config}", root / "inference.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    classifier = module.CitationClassifier(root)
    # Las exportaciones de la primera corrida dejaban la atención en automático.
    # Homogeneizarla en memoria, sin modificar inference.py ni los pesos guardados.
    if classifier.model.config._attn_implementation != "eager":
        del classifier.model
        classifier.model = AutoModelForSequenceClassification.from_pretrained(
            root, local_files_only=True, attn_implementation="eager").float().to(classifier.device).eval()
    return classifier

def predict_encoded(classifier, data):
    return predict_export_fp32(classifier, data, EVAL_BATCH)

def report_predictions(config, split, rows, probs, evaluation_run):
    y = np.asarray([LABEL2ID[r["label"]] for r in rows])
    pred = probs.argmax(-1)
    report = classification_report(y, pred, labels=list(ID2LABEL), target_names=LABELS,
                                   output_dict=True, zero_division=0)
    root = EVALUATION_ROOT / config
    root.mkdir(parents=True, exist_ok=True)
    save_json(root / f"{split}_classification_report.json", report)
    metrics = {"config": config, "split": split, "n": len(rows),
               "evaluation_policy": "v2_fp32", "precision": "fp32",
               **compute_metrics(type("Prediction", (), {"predictions": probs, "label_ids": y})())}
    save_json(root / f"{split}_scores.json", metrics)
    frame = pd.DataFrame({"inst_key": [r["inst_key"] for r in rows],
                          "gold_or_prelabel": [r["label"] for r in rows],
                          "label_source": [r["label_source"] for r in rows],
                          "predicted_label": [ID2LABEL[int(x)] for x in pred]})
    for i, label in ID2LABEL.items():
        frame[f"p_{label}"] = probs[:, i]
    frame.to_csv(root / f"{split}_predictions.csv", index=False)
    frame.loc[frame.gold_or_prelabel != frame.predicted_label].to_csv(root / f"{split}_errors.csv", index=False)
    per_class = pd.DataFrame([{"label": label, **report[label]} for label in LABELS])
    per_class.to_csv(root / f"{split}_per_class.csv", index=False)
    display(per_class)
    for index, label in ID2LABEL.items():
        tracking.metrics(evaluation_run, {f"{split}/class_{index}/{key}": value
            for key, value in report[label].items()})
    cm = confusion_matrix(y, pred, labels=list(ID2LABEL))
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(root / f"{split}_confusion.csv")
    short = ["BG", "Gap", "Basis", "Comp", "App", "I/M", "Evid", "Origin", "FR"]
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set(xticks=range(9), yticks=range(9), xticklabels=short, yticklabels=short,
           xlabel="Predicción", ylabel="Referencia", title=f"{config} · {split}")
    for i in range(9):
        for j in range(9):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(root / f"{split}_confusion.png", dpi=160)
    plt.show()
    plt.close(fig)
    tracking.metrics(evaluation_run, {f"{split}_fp32_{k}": v for k, v in metrics.items()})
    tracking.metrics(tracking.parent(config), {f"{split}_fp32_{k}": v for k, v in metrics.items()})
    return metrics

val_summary = []
for config in CONFIGS:
    evaluation_dir = EVALUATION_ROOT / config
    evaluation_dir.mkdir()
    with tracking.evaluation(config, evaluation_dir) as evaluation_run:
        clf = load_export(config)
        try:
            sample = clf.predict(texts[val_rows[0]["inst_key"]])
            single = predict_encoded(clf, encoded[config]["validation"].select([0]))[0]
            assert np.allclose(single, [sample["scores"][k] for k in LABELS], atol=2e-5)
            training_metrics = json.loads((OUTPUT_ROOT / config / "validation_metrics.json").read_text())
            precision = "bf16" if USE_BF16 else "fp16"
            probs, trainer_probs, audit = audit_export(clf, encoded[config]["validation"],
                [r["inst_key"] for r in val_rows], training_metrics.get("best_checkpoint"),
                evaluation_dir, EVAL_BATCH, SEED, precision)
            replayed = probability_metrics([LABEL2ID[r["label"]] for r in val_rows], trainer_probs)
            audit.update({"run_id": RUN_ID, "config": config, "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION, "split_hashes": split_hashes, "labels": LABELS,
                "label_sources": dict(collections.Counter(r["label_source"] for r in val_rows)),
                "training_complete_sha256": sha256(OUTPUT_ROOT / config / "TRAINING_COMPLETE.json"),
                "recorded_training_f1_macro": training_metrics.get("eval_f1_macro"),
                "replayed_trainer_metrics": replayed})
            save_json(evaluation_dir / "export_precision_audit.json", audit)
            tracking.client.set_tag(evaluation_run, "export_check", audit["export_check"])
            tracking.client.set_tag(evaluation_run, "trainer_precision", precision)
            tracking.client.set_tag(evaluation_run, "selected_checkpoint", str(training_metrics.get("best_checkpoint")))
            tracking.client.set_tag(evaluation_run, "model_revision", MODEL_REVISION)
            tracking.metrics(evaluation_run, {f"validation_trainer_replay_{k}": v for k, v in replayed.items()})
            tracking.metrics(evaluation_run, {"precision_label_disagreements": audit["trainer_vs_export_fp32"]["label_disagreements"]})
            val_summary.append(report_predictions(config, "validation", val_rows, probs, evaluation_run))
            print("Auditoría:", audit["export_check"], "cambios de etiqueta por precisión:",
                  audit["trainer_vs_export_fp32"]["label_disagreements"])
        finally:
            del clf
            gc.collect()
            torch.cuda.empty_cache()
pd.DataFrame(val_summary).to_csv(EVALUATION_ROOT / "validation_comparison.csv", index=False)
if {"base", "fragmentos"} <= {row["config"] for row in val_summary}:
    scores = {row["config"]: row["f1_macro"] for row in val_summary}
    print("Delta F1 macro de validación (fragmentos - base):", scores["fragmentos"] - scores["base"])
print("Evaluación versionada:", EVALUATION_ROOT)
display(pd.DataFrame(val_summary))


## 15. Dejamos preparada la evaluación final

Aquí distinguimos las dos mediciones. La validación interna usa etiquetas
automáticas y sirve para elegir el mejor punto de entrenamiento. La prueba final
usa las 2.968 citas revisadas por humanos y mostrará cómo responden los modelos
frente a esa referencia.

Esta corrida completa entrenamiento y validación interna. El siguiente paso de A4
es ejecutar esa prueba con los modelos ya fijados, sin reajustarlos después con
sus resultados. Improvement/Modification tiene allí 268 casos.


In [ ]:
test_summary = []
test_probabilities = {}
assert not RUN_TEST
print("T_B sigue reservado. Este notebook termina con entrenamiento y validación, sin medir el test humano.")
print("Cierre A3:", a3_closure["valid_test_rows"], "casos válidos; se usarán sus cupos reales, sin completar clases artificialmente.")


## 16. Guardamos los modelos y el respaldo final

Vamos a empaquetar cada modelo con su tokenizador y las instrucciones para preparar
las entradas. Esto permitirá que los compañeros lo utilicen sin repetir el entrenamiento.
También dejamos en MLflow las gráficas, tablas y rutas de Drive.

Cuando termine esta celda y el notebook quede guardado, podemos finalizar el entorno
de Colab para liberar la A100. Al terminar tendremos los modelos y las métricas de validación interna
guardados. La evaluación final se hará con esos modelos ya fijados.


In [ ]:
for config in CONFIGS:
    model_dir = OUTPUT_ROOT / config / "model"
    archive = OUTPUT_ROOT / f"scibert_g15_{config}.zip"
    if not archive.exists():
        archive = shutil.make_archive(str(archive.with_suffix("")), "zip", model_dir)
    print("Modelo:", archive)
print("Artefactos privados:", OUTPUT_ROOT)
for config in CONFIGS:
    tracking.reports(config)
    tracking.client.set_tag(tracking.parent(config), "model_zip_drive", str(OUTPUT_ROOT / f"scibert_g15_{config}.zip"))
    tracking.client.set_tag(tracking.parent(config), "latest_evaluation_drive", str(EVALUATION_ROOT / config))
    for path in sorted(EVALUATION_ROOT.glob("*.csv")) + sorted(EVALUATION_ROOT.glob("*.json")) + sorted(EVALUATION_ROOT.glob("*.png")):
        tracking.client.log_artifact(tracking.parent(config), str(path), artifact_path="evaluation_comparisons/" + EVALUATION_ROOT.name)
tracking.backup(force=True)
print("Terminamos entrenamiento y validación. Modelos y respaldo guardados; T_B sigue sin medirse.")
